# Dimer Confidence Metric Diagnostic Notebook

**Purpose:** This notebook helps you understand *why* different confidence scores for a predicted two-chain protein complex from the AlphaFold Database (AFDB) agree or disagree.

**Which complexes does it cover?** Any AFDB dimer. That means a **homodimer**, whose two chains are copies of one protein, or a **heterodimer**, whose two chains are different proteins and usually different lengths. There is nothing to switch between the two: the notebook reads each chain's identity and length from the model and labels every table, figure and 3D view accordingly.

**What are confidence scores?** AlphaFold and related tools compute several numbers that estimate how reliable the predicted structure is. This notebook computes five of them:
- **ipTM** – Overall confidence in chain positioning
- **ipSAE** – Stricter confidence using only well-predicted residues
- **pDockQ** – Structural plausibility (contact count + pLDDT)
- **pDockQ2** – Like pDockQ but also checks PAE at the interface
- **LIS** – Density of low-error inter-chain interactions

**No structural biology background required.** Every section includes a plain-language explanation before any code.

---
**Usage:** Set `ACCESSION_ID` in Section 1, then run all cells (Runtime → Run all).

In [ ]:
# Bootstrap: this one cell works unchanged locally and on Google Colab.
#
# Locally it finds the checkout you are already running from, and touches
# nothing: no clone, no fetch, no reset. On Colab it shallow-clones the public
# repo into /content, refreshing that clone if it is already there. Either way
# it puts `src/` on sys.path and imports the analysis module from there.
#
# Deliberately NOT `pip install`-ing InsightFold itself: that would resolve
# pyproject.toml and drag in biopython, gemmi, scipy and plotly, breaking both
# the project's dependency rules and the 60 s Colab install budget.
# The repo is public, so there is no token, no auth header and no getpass
# (getpass would block forever in a Run-all notebook).
import shutil
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path

REPO_URL = 'https://github.com/PDBeurope/InsightFold.git'

# TODO(merge): revert REPO_BRANCH to 'main' (or a release tag) once the
# homodimer-notebook-rework branch merges. `complex_interface_utils` exists only
# on that branch today, so cloning 'main' gives a checkout without it. Grep for
# "TODO(merge)" before releasing this notebook.
REPO_BRANCH = 'homodimer-notebook-rework'

COLAB_CLONE_DIR = Path('/content/InsightFold')

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_repo_root(start=None):
    """Walk up from `start` (default: cwd) for a dir holding both pyproject.toml and src/."""
    s = Path(start) if start is not None else Path.cwd()
    s = s.expanduser().resolve()
    for d in (s, *s.parents):
        if (d / 'pyproject.toml').is_file() and (d / 'src').is_dir():
            return d
    return None


def describe_checkout(root):
    """'branch @ sha' for a git checkout, or a plain note when it is not one."""
    try:
        rev = subprocess.run(['git', '-C', str(root), 'rev-parse', '--abbrev-ref', 'HEAD'],
                             capture_output=True, text=True)
        sha = subprocess.run(['git', '-C', str(root), 'rev-parse', '--short', 'HEAD'],
                             capture_output=True, text=True)
        if rev.returncode == 0 and sha.returncode == 0:
            return f'{rev.stdout.strip()} @ {sha.stdout.strip()}'
    except (OSError, subprocess.SubprocessError):
        pass
    return 'unknown (not a git checkout)'


def ensure_colab_clone(clone_dir, url=REPO_URL, branch=REPO_BRANCH):
    """Clone `url`@`branch` into `clone_dir`, or refresh a clone already there.

    Only ever called with the disposable Colab scratch directory: `git reset
    --hard` must never run against a checkout somebody is working in. A clone
    left over from an earlier session can predate the branch this notebook
    needs, which is why an existing clone is refreshed rather than reused.
    """
    clone_dir = Path(clone_dir)

    if clone_dir.exists() and not (clone_dir / '.git').is_dir():
        # Something is in the way that is not a clone. Move it aside rather than
        # delete it, so nothing of the user's is destroyed silently.
        aside = clone_dir.with_name(clone_dir.name + '.not-a-git-repo')
        shutil.rmtree(aside, ignore_errors=True)
        print(f'{clone_dir} exists but is not a git clone; moving it to {aside}')
        clone_dir.rename(aside)

    if not (clone_dir / '.git').is_dir():
        print(f'Cloning {url} ({branch}) ...')
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', branch, url, str(clone_dir)],
            check=True,
        )
        return

    print(f'Refreshing existing clone at {clone_dir} -> {branch} ...')

    def _git(*args):
        return subprocess.run(['git', '-C', str(clone_dir), *args],
                              capture_output=True, text=True)

    for step in (('fetch', '--depth', '1', 'origin', branch),
                 ('reset', '--hard', 'FETCH_HEAD')):
        done = _git(*step)
        if done.returncode != 0:
            # Offline, or a clone we cannot reason about: an existing checkout is
            # better than a hard failure, but say so loudly.
            detail = (done.stderr or done.stdout).strip().splitlines()
            print(f'  git {step[0]} failed; falling back to the existing checkout as-is.')
            if detail:
                print(f'  git said: {detail[-1]}')
            return
    print(f'  now at {describe_checkout(clone_dir)}')


# Search from the working directory only. The Colab clone is handled separately
# below so that a stale clone gets refreshed instead of being picked up as-is.
REPO_ROOT = find_repo_root()

if IN_COLAB and (REPO_ROOT is None or REPO_ROOT == COLAB_CLONE_DIR.resolve()):
    ensure_colab_clone(COLAB_CLONE_DIR)
    REPO_ROOT = find_repo_root(COLAB_CLONE_DIR)

if REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the InsightFold checkout.\n'
        f'Looked upwards from {Path.cwd()} for a directory containing both '
        "'pyproject.toml' and 'src/'.\n"
        'Run this notebook from inside a clone of '
        'https://github.com/PDBeurope/InsightFold, for example:\n'
        f'    git clone -b {REPO_BRANCH} https://github.com/PDBeurope/InsightFold.git\n'
        '    cd InsightFold && jupyter lab notebooks/homodimer_diagnostic.ipynb'
    )

SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# molviewspec powers the optional 3D views in Section 6. Install it only when it
# is genuinely missing, and only on Colab: locally it comes from the environment,
# and a pip call on every run is pure latency.
if find_spec('molviewspec') is None and IN_COLAB:
    print('Installing molviewspec ...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'molviewspec'],
        check=False,
    )

HAS_MOLVIEWSPEC = find_spec('molviewspec') is not None

REVISION = describe_checkout(REPO_ROOT)

try:
    from insightfold import complex_interface_utils as ciu  # noqa: E402
except ImportError as exc:
    _remedy = (
        f'On Colab, remove the clone and re-run this cell to fetch a fresh copy:\n'
        f'    !rm -rf {COLAB_CLONE_DIR}\n'
        if IN_COLAB else
        f'Locally, update your checkout:\n'
        f'    git -C {REPO_ROOT} switch {REPO_BRANCH} && git -C {REPO_ROOT} pull\n'
    )
    raise ImportError(
        f"Could not import 'insightfold.complex_interface_utils': {exc}\n"
        f'  repo root in use: {REPO_ROOT}\n'
        f'  revision in use:  {REVISION}\n'
        f'  branch expected:  {REPO_BRANCH}\n'
        'The likely cause is a checkout that predates this module or sits on a '
        f"different branch: complex_interface_utils.py lives on '{REPO_BRANCH}'.\n"
        + _remedy
    ) from exc

print(f'Environment: {"Colab" if IN_COLAB else "local"}')
print(f'Repo root:   {REPO_ROOT}')
print(f'Revision:    {REVISION}')
print(f'molviewspec: {"available" if HAS_MOLVIEWSPEC else "not installed (Section 6 will be skipped)"}')

In [ ]:
%matplotlib inline
import json

import numpy as np
from IPython.display import HTML, display

# Every formula, threshold, figure and 3D view now lives in the module the
# bootstrap cell imported as `ciu`; this notebook keeps only the narrative and
# the orchestration. Styling is an explicit call because importing a module must
# never restyle somebody else's figures as a side effect.
ciu.apply_plot_style()

print('Imports OK')


---
## Section 1 — Setup and Data Loading

**What happens here?**
We fetch all the raw data from the AlphaFold Database (AFDB) for one dimer accession, homodimer or heterodimer alike.

The AFDB stores:
- The **3D structure** in mmCIF format (atom positions for every residue)
- The **PAE matrix** — a grid of numbers where entry `[i, j]` = how uncertain AlphaFold is about the position of residue `i` given that residue `j` is aligned correctly (lower = better)
- The **pLDDT scores** — a per-residue confidence score (0–100, higher = better)

We parse all three into data structures we can analyse.

**What if the accession is not a dimer?** The notebook stops and says so. Every score below is an *inter-chain* measurement, so a monomer has nothing to measure and anything with more than two chains has no single pair to measure. A mistyped accession, a monomer or a larger assembly each produce a short message naming what was found, what is supported and what to do next — not a stack trace and not a misleading zero.

In [ ]:

# ── USER INPUT ────────────────────────────────────────────────────────
ACCESSION_ID = 'AF-0000000065889468'  # @param {type:"string"}

# Which direction of the chain pair to inspect (R008). PAE is asymmetric, so
# every inter-chain score is measured twice, once from each chain's frame of
# reference. This chooses which of the two the directional report highlights and
# which panel the per-residue profile figure draws:
#   None   both directions, plus the combination ipsae.py reports (the default)
#   'xy'   the first chain of the ordered pair read against the second
#   'yx'   the second read against the first
# It is a *viewing* control and nothing else. No score is recomputed, and no
# reported value, traffic light or summary-table entry can change with it,
# because ipsae.py defines each score as a combination of both directions.
# The two chain ids are also accepted, e.g. 'AB' or 'B->A'; a typo is rejected
# in Section 4, where the real chain ids are known.
DIRECTION = None

# Colour scale for every PAE figure (R051). Dark always means low PAE, i.e.
# confident; only the hue is a choice, so switching palettes cannot invert the
# reading of a figure. One assignment retunes Section 3 and everything else that
# draws PAE, because every plot resolves `ciu.PAE_CMAP` at draw time.
#   'green'            sequential green, the default. Lightness falls
#                      monotonically along the whole ramp under normal vision
#                      and under simulated deuteranopia and protanopia, so the
#                      scale is readable without colour discrimination.
#   'colourblind_safe' viridis: perceptually uniform, dark violet = low PAE,
#                      the largest step-to-step separation of the three under
#                      simulated colour-vision deficiency.
#   'rdbu'             the pre-M5 diverging blue-white-red. Kept so an older
#                      figure can be reproduced; not recommended, because PAE
#                      has no meaningful midpoint and this map's lightness is
#                      non-monotone, so a very confident cell and a hopeless
#                      one can render at the same lightness.
ciu.PAE_CMAP = 'green'
# ───────────────────────────────────────────────────────────────────

# Set True to upload local files instead of fetching from AFDB
USE_LOCAL_FILE = False

# The cutoffs are the module's, so the scores, the figures and the printed
# summaries can never disagree about which value is in force.
PAE_CUTOFF  = ciu.PAE_CUTOFF    # 10.0 Å — AFDB standard cutoff for ipSAE
DIST_CUTOFF = ciu.DIST_CUTOFF   #  8.0 Å — CB-CB contact cutoff
LIS_CUTOFF  = ciu.LIS_CUTOFF    # 12.0 Å — PAE cutoff for LIS


In [ ]:
import ipywidgets as _widgets
from IPython.display import display as _display

if USE_LOCAL_FILE:
    # All three uploads are required, not one plus two optionals (R025). Six of
    # the seven values are read off the PAE matrix and the seventh, pDockQ, needs
    # per-residue pLDDT, so a partial upload leaves the traffic light with
    # nothing to colour. The next cell refuses a partial upload by name, instead
    # of the skip this cell used to advertise and never perform.
    _upload_cif   = _widgets.FileUpload(accept='.cif,.mmcif', multiple=False,
                                        description='mmCIF (required)')
    _upload_pae   = _widgets.FileUpload(accept='.json',       multiple=False,
                                        description='PAE JSON (required)')
    _upload_plddt = _widgets.FileUpload(accept='.json',       multiple=False,
                                        description='pLDDT JSON (required)')
    _display(_widgets.VBox([
        _widgets.HTML(
            '<b>Upload all three files, then run the next cell.</b><br>'
            'All three are required: the PAE matrix supplies six of the seven '
            'scores, and per-residue pLDDT supplies the seventh.<br>'
            'For an AFDB model they are the <code>cifUrl</code>, '
            '<code>paeDocUrl</code> and <code>plddtDocUrl</code> downloads — '
            '<code>…-model_v1.cif</code>, '
            '<code>…-predicted_aligned_error_v1.json</code> and '
            '<code>…-confidence_v1.json</code>.'),
        _upload_cif,
        _upload_pae,
        _upload_plddt,
    ]))
else:
    print('Online mode — files will be downloaded from AFDB.')

In [ ]:
if USE_LOCAL_FILE:
    prediction = None
    print(f'Local file mode — skipping AFDB API for {ACCESSION_ID}')
    # With no metadata there is nothing for the declared-assembly gate below to
    # read, so local-file mode passes only one of the two. The one it passes is
    # the structural gate inside `verify_chain_identity`, which is the
    # authoritative one: it reads the chains themselves, and a monomer or a
    # three-chain upload is refused there with the same message either way.
    print('No metadata to check, so the declared-assembly gate is skipped here. '
          'The\nstructural gate inside verify_chain_identity still runs, and is '
          'what refuses a\nmonomer or a model with more than two chains.')
else:
    print(f'Fetching: {ciu.AFDB_PREDICTION_URL.format(accession=ACCESSION_ID)}')
    prediction = ciu.fetch_afdb_metadata(ACCESSION_ID)
    # One entry per chain, and the endpoint's order is non-deterministic: the
    # same accession answers ['A', 'B'] on one call and ['B', 'A'] on the next.
    # `AFDBPrediction` sorts the entries by chain id at construction and every
    # field below is read by chain id, so nothing indexes an entry by position.
    print(f'Chains described: {list(prediction.chain_ids)}')
    print('Available fields:', sorted({_field
                                       for _chain in prediction.chain_ids
                                       for _field in prediction.entry_for_chain(_chain)}))

    # First of two assembly gates. This one reads only what AFDB *declares* —
    # `assemblyType`, `oligomericState`, `complexComposition`, `isComplex` — and
    # so can refuse a monomer here, before three downloads. It is deliberately
    # not the last word: the endpoint can describe fewer chains than the model
    # actually has, so it refuses only when the declaration and the endpoint's
    # own per-chain entry list agree that this is not a two-chain complex. A
    # declaration the entries contradict is reported as a conflict, not acted
    # on. The gate that always fires is the structural one below, inside
    # `verify_chain_identity`, which nothing downstream can bypass.
    _declared = ciu.describe_assembly(prediction).require_dimer()
    print('Declared assembly:', _declared.declared_phrase)


In [ ]:
def _read_upload(widget):
    """Return bytes from a FileUpload widget, or None if nothing was uploaded."""
    if not widget.value:
        return None
    return bytes(widget.value[0]['content'])


def _upload_name(widget):
    """The uploaded file's name, or a marker saying the slot is empty."""
    return widget.value[0]['name'] if widget.value else '(not uploaded)'


if USE_LOCAL_FILE:
    _cif_bytes   = _read_upload(_upload_cif)
    _pae_bytes   = _read_upload(_upload_pae)
    _plddt_bytes = _read_upload(_upload_plddt)

    print(f'mmCIF : {_upload_name(_upload_cif)}')
    print(f'PAE   : {_upload_name(_upload_pae)}')
    print(f'pLDDT : {_upload_name(_upload_plddt)}')

    # R025. This cell used to print "PAE file not uploaded — PAE-dependent
    # analyses will be skipped" and then skip nothing: `pae_raw` stayed None and
    # `parse_pae(None)` raised a bare TypeError two cells further on. There is no
    # useful partial run to skip *to* — without PAE and pLDDT every score but a
    # contact count is gone — so the honest answer is to refuse here, while the
    # upload widget is still on screen, naming every missing file at once.
    ciu.require_local_documents(_cif_bytes, _pae_bytes, _plddt_bytes,
                                accession=ACCESSION_ID)

    cif_text  = _cif_bytes.decode('utf-8', errors='replace')
    pae_raw   = json.loads(_pae_bytes)
    plddt_raw = json.loads(_plddt_bytes)
    print('All three local files loaded.')
else:
    print(f'Downloading mmCIF: {prediction.cif_url}')
    cif_text  = ciu.download_structure(prediction)
    print(f'Downloading PAE:   {prediction.pae_url}')
    pae_raw   = ciu.download_pae(prediction)
    print(f'Downloading pLDDT: {prediction.plddt_url}')
    plddt_raw = ciu.download_plddt(prediction)
    print('All downloads complete.')

In [ ]:
# mmCIF parsing, CB/CA selection and the GLY fallback all live in the module.
chains = ciu.parse_structure(cif_text)
chain_ids = sorted(chains)

print(f'Chains found: {chain_ids}')
for _chain_id in chain_ids:
    print(f'  Chain {_chain_id}: {chains[_chain_id].n_residues} residues')


In [ ]:
_structure_lengths = {cid: chain.n_residues for cid, chain in chains.items()}
pae   = ciu.parse_pae(pae_raw, fallback_lengths=_structure_lengths)
plddt = ciu.parse_plddt(plddt_raw, fallback_lengths=_structure_lengths)

# Three sources describe the chains — the mmCIF, the PAE document and the pLDDT
# document — and every quadrant slice below assumes all three agree. A
# disagreement misaligns the slices and yields plausible but wrong scores rather
# than an error, so agreement is a checked precondition, tested on all three
# edges of the triangle. Chain ids are never paired positionally: an A/C
# structure against an A/B document fails loudly instead of being guessed at.
# The same call resolves each chain's real protein identity for use as a label,
# and reconciles what AFDB says the assembly is against what the chains actually
# are. It is also the notebook's dimer gate: a monomer or a three-chain model is
# refused here, with an explanation, instead of failing on `chain_ids[1]` below.
identity = ciu.verify_chain_identity(chains, pae, plddt, prediction)

print(identity.assembly.headline)
print()
print('Chains verified across structure, PAE and pLDDT:')
print(identity.legend())
for _note in identity.notes:
    print(f'NOTE: {_note}')

# D4: everything downstream takes one explicit ordered chain pair. Nothing here
# hard-codes 'A' and 'B'; PAE is asymmetric, so the order is part of the query.
chain_x, chain_y = pae.chain_ids[0], pae.chain_ids[1]
pair    = pae.ordered_pair(chain_x, chain_y)
plddt_x = plddt.for_chain(chain_x)
plddt_y = plddt.for_chain(chain_y)
nx, ny  = pair.nx, pair.ny

# Two label forms per chain: the compact one for axes, ticks and tabular output,
# the full one for captions and headings. Both keep the chain id, so a
# homodimer's two halves stay distinguishable even though the protein is one.
label_x, label_y = identity.label(chain_x), identity.label(chain_y)
short_x, short_y = label_x.short, label_y.short
full_x,  full_y  = label_x.full,  label_y.full

print(f'\nPAE matrix shape: {pae.matrix.shape}')
print('PAE quadrants:')
print(f'  intra {short_x}: {pair.block_xx.shape}')
print(f'  inter {short_x} → {short_y}: {pair.block_xy.shape}')
print(f'  inter {short_y} → {short_x}: {pair.block_yx.shape}')
print(f'  intra {short_y}: {pair.block_yy.shape}')
print(f'pLDDT mean: {short_x} {plddt_x.mean():.1f}, {short_y} {plddt_y.mean():.1f}')


In [ ]:
# UniProt accession, protein name, gene, organism and monomer length describe a
# *chain*, not the complex — the endpoint returns one entry per chain — so they
# are reported per chain and attributed. Chains that are the same protein
# collapse into one block, so a homodimer does not repeat itself and a
# heterodimer shows both proteins, with no switch between the two cases (D3).
#
# The Assembly / Composition / AFDB-declares rows carry two independent kinds of
# evidence: what the metadata asserts, and what the chains themselves show. They
# are printed side by side and any disagreement is spelled out under a `[!]`,
# because either source can be the wrong one and only the reader can judge which.
# The same `AssemblyDescription` the dimer gate used is passed in, so the report
# and the check that let the run get this far cannot drift apart.
print(ciu.format_metadata_report(prediction,
                                 {chain_x: nx, chain_y: ny},
                                 accession=ACCESSION_ID,
                                 assembly=identity.assembly,
                                 labels=identity.labels))


---
## Section 2 — Interface Detection

**What is the interface?**
The interface is the region where the two protein chains touch each other. We identify it by measuring distances between atoms: if a residue in one chain has a beta-carbon (CB) within **8.0 Å** of a CB atom in the other chain, those two residues are in "contact" and are part of the interface.

*(We use the beta-carbon CB because it better represents the side chain position. For glycine, which has no CB, we use the alpha-carbon CA instead.)*

**Why does this matter?**
The interface residues are the ones where the two chains actually interact. Many confidence metrics focus specifically on these residues — if AlphaFold is uncertain about the interface, that's a red flag.

**Two pictures of one measurement.**
This section works out a single set of interface residues and then draws it twice: as a contact map with per-chain coverage tracks, and as a 3D view of the structure itself. The second figure exists so that the pattern in the first can be checked against physical reality.

In [ ]:
contacts = ciu.detect_interface(chains[chain_x], chains[chain_y],
                                dist_cutoff=DIST_CUTOFF)

print(f'Contact cutoff         : {contacts.dist_cutoff} Å (CB-CB; CA for GLY)')
print(f'Number of contact pairs: {contacts.n_contact_pairs}')
print(f'Interface residues, {short_x}: {contacts.n_interface_residues_x} / {nx} '
      f'({100 * contacts.n_interface_residues_x / nx:.1f}%)')
print(f'Interface residues, {short_y}: {contacts.n_interface_residues_y} / {ny} '
      f'({100 * contacts.n_interface_residues_y / ny:.1f}%)')

# Both chains, not just the first: for a heterodimer the two ranges are
# genuinely different numbers, and reporting one of them was only ever harmless
# while the chains were copies of each other.
for _chain_id, _short, _mask in ((chain_x, short_x, contacts.mask_x),
                                 (chain_y, short_y, contacts.mask_y)):
    _if_res = chains[_chain_id].res_ids[_mask]
    if _if_res.size:
        print(f'Interface residue range, {_short}: {_if_res[0]} – {_if_res[-1]}')


In [ ]:
display(ciu.plot_interface_contact_map(contacts, short_x, short_y))


### View 1 — the same interface in 3D

The contact map above and the viewer below show **the same residues in two different coordinate systems**, and nothing is recomputed between them: both read `contacts.mask_x` and `contacts.mask_y`, the two masks the counts above were printed from.

- The **map** is indexed by residue number. A contact is a square at (row = a residue of the first chain, column = a residue of the second), shaded by the CB-CB distance that qualified it, dark being the closer contact. The two coverage tracks beside it mark in amber which residue indices along each chain appear anywhere in that block.
- The **3D view** takes those same residues and puts them back where they physically are.

**What is drawn.** Both chains as cartoon ribbons, plus ball-and-stick side chains for the interface residues of *both* chains. The view is interactive: drag to rotate, scroll to zoom, click a residue for its identity.

**What the colours encode.** Identity, not confidence. Nothing here is a score, and no PAE value has been read yet. The first chain's cartoon is teal and the second's is coral; interface side chains are gold on the first chain and cornflower on the second, each chosen to stay legible against its own chain's cartoon.

**What you should be able to check by looking at both.**

- The legend under the viewer carries the interface residue count per chain. It should equal the number of amber residues in that chain's coverage track, and the count printed above the map. The two figures are the same measurement or one of them is wrong.
- A **contiguous amber stretch** should appear as **one patch of sticks**: residues adjacent in sequence are adjacent in space along a helix or a strand.
- **Separate amber stretches need not be separate patches.** Two segments far apart in sequence can fold onto the same face, and then several blocks in the map are one surface in 3D. That is the check the map cannot perform on its own, and the reason for drawing both.
- The **residue range** printed above is an outer bound on the patch, not a measure of its size. On a strongly asymmetric pair the range can span most of a long chain while only a couple of dozen residues inside it are amber, because the patch is assembled from loops that sit far apart in sequence. Read the count, and look at the structure; a handful of interface residues on a long chain is what a low pDockQ looks like before any score has been computed (section 4.3).

**What a problem looks like.** Sticks scattered over genuinely disconnected surfaces of the protein rather than one contiguous face, or an interface made of a handful of residues on a long chain. Both are geometry that Section 4 will later read as a weak interface, so seeing it here means the low score that follows is not a surprise.

This is the first of six 3D views. The other five colour this same structure by a score and are in **Section 6**, once those scores exist. The numbering runs continuously across the two sections, so a later reference to "View 1" points here.

In [ ]:
# The 3D setup lives in Section 2 because this is now the first section to draw
# a view. `_source` and `render_view` are reused unchanged by Views 2 to 6 in
# Section 6, so the structure URL is resolved once and the two sections cannot
# end up pointing Mol* at different files.
_source = None
if not ciu.molviewspec_available():
    print(ciu.MOLVIEWSPEC_MISSING_MESSAGE)
elif prediction is None:
    print('Skipping the 3D views: Mol* downloads the structure itself, so '
          'local file mode has no URL to hand it.\nThe contact map above, and '
          'every score in this notebook, are unaffected.')
else:
    try:
        _source = ciu.resolve_structure_source(prediction)
    except ValueError as _exc:
        print(f'Skipping the 3D views: {_exc}')


def render_view(key, build, legend):
    """Draw one view with its caption and legend, or say why it was skipped.

    Built lazily and caught per view, so one failing viewer does not cost you
    the others.
    """
    label = ciu.format_view_label(key, short_x, short_y)
    if _source is None:
        print(f'{label}\n  skipped: no structure source resolved above.')
        return
    try:
        ciu.show_mol_view(build(), label)
        display(HTML(ciu.legend_html(legend())))
    except Exception as exc:
        print(f'{label} failed: {exc}')


render_view('chain_overview',
            lambda: ciu.build_chain_overview_view(_source, chains, contacts),
            lambda: ciu.chain_overview_legend(contacts, short_x, short_y))

---
## Section 3 — PAE Matrix Decomposition

**What is the PAE matrix?**
The Predicted Aligned Error (PAE) matrix is an N×N grid where N = total number of residues in both chains combined. Entry `[i, j]` = AlphaFold's estimate of how wrong the position of residue `i` would be, *if* we fixed residue `j` and rotated/translated everything else to align it.

Every PAE figure in this notebook uses a **sequential colour scale on which dark means low PAE**. The hue is a setting; the direction is not, so a palette switch cannot invert how a figure reads.

- **Dark, saturated cells** → low PAE → AlphaFold is confident about the relative position
- **Pale, washed-out cells** → high PAE → AlphaFold is uncertain

Confidence is therefore what carries the ink, and uncertainty recedes toward the page. Read the colour bar rather than the hue: its numbers are Ångströms, and the scale runs from 0 at the dark end.

**Changing the palette.** Set `ciu.PAE_CMAP` in the user-input cell in Section 1. `'green'` is the default; `'colourblind_safe'` is viridis, for a reader who would rather not rely on a single hue; `'rdbu'` is the diverging blue-white-red this notebook used before, kept only so an older figure can be reproduced. All three put dark at PAE 0.

The chain boundary splits the matrix into four blocks. Writing the first chain of the pair as A and the second as B (the figures below use each chain's real name):
- **Top-left (AA)**: confidence within the first chain
- **Bottom-right (BB)**: confidence within the second chain
- **Top-right (AB)** and **bottom-left (BA)**: confidence *between* chains — this is what all inter-chain scores use

The four blocks are equally sized only when the two chains are: for a heterodimer, AA is nA×nA, BB is nB×nB, and the two inter-chain blocks are nA×nB and nB×nA. AB and BA are also not transposes of one another, because PAE is asymmetric, so each inter-chain score is computed in both directions.

Each score uses a different subset of the inter-chain region. **The second figure below zooms in on the top-right (AB) block only** — one quadrant of the matrix in the first figure, not the whole thing — and shows which of its cells each score reads. Its own caption states the slice and marks the quadrant on a thumbnail of the full matrix, so the two figures can be lined up without counting residues.

In [ ]:
display(ciu.plot_pae_matrix(pae, chain_x, chain_y, accession=ACCESSION_ID,
                            labels=identity.labels))


In [ ]:
display(ciu.plot_pae_score_masks(pair, contacts, max_pae=pae.max_pae,
                                 pae_cutoff=PAE_CUTOFF, lis_cutoff=LIS_CUTOFF,
                                 label_x=short_x, label_y=short_y))

# Same masks the panels are drawn from, so the figure and the numbers agree.
_masks = ciu.score_masks(pair, contacts,
                         pae_cutoff=PAE_CUTOFF, lis_cutoff=LIS_CUTOFF)
_block_cells = pair.block_xy.size
print(f'Cells of the {short_x} → {short_y} inter-chain block '
      f'(pae_matrix[:{nx}, {nx}:{nx + ny}]) used by each score:')
for _name, _mask in _masks.items():
    print(f'  {_name:12s}: {int(_mask.sum()):6d} cells '
          f'({100 * int(_mask.sum()) / _block_cells:.1f}%)')


---
## Section 4 — Score Computation

**What we do here:** compute the seven confidence values, one at a time. Each score gets its own subsection with the same four parts before any code runs:

1. **What it measures**, in plain language.
2. **The formula**, exactly as `ipsae.py` computes it.
3. **Why its authors built it that way**, from the paper that introduced it.
4. **How to read the number**, with the threshold and where the threshold came from.

**Seven values, five ideas.** ipSAE appears three times because it has three normalisation variants. They are kept together in 4.2 rather than split across three subsections, because the only thing separating them is one input to one function, and splitting them would hide exactly that.

**One shared piece of machinery.** Four of the five ideas begin by turning a PAE value into a score between 0 and 1 with the same transform, which comes from the TM-score of Zhang and Skolnick:

```
ptm(x, d0) = 1 / (1 + (x / d0)²)
```

A PAE of 0 Å gives 1.0, a PAE equal to `d0` gives 0.5, and the score decays from there. The scale parameter `d0` is a function of a residue count `L`:

```
d0(L) = max(1.0,  1.24 × (L - 15)^(1/3) - 1.8)
```

(Dunbrack 2025, p. 4 Eq. 2 and p. 9 Eq. 15.) A **larger `L` gives a larger `d0`, and a larger `d0` is more forgiving**: the same PAE value scores higher. That single sentence explains most of the disagreements you are about to see, because the scores below differ mainly in what they count as `L`.

**A note on the code.** Every formula here is transcribed from `DunbrackLab/IPSAE` `ipsae.py` v4, the reference implementation, and agrees with it to within 5e-5 on the fixtures. The notebook calls the module rather than defining the maths inline, so there is exactly one copy of each formula in the project.

**Reading order.** 4.1 ipTM_d0chn · 4.2 ipSAE (d0res, d0chn, d0dom) · 4.3 pDockQ · 4.4 pDockQ2 · 4.5 LIS · 4.6 all seven together, plus the directional breakdown and the per-residue profiles.

### 4.1 ipTM_d0chn — the baseline everything else improves on

**What it measures.** How confident AlphaFold is about where one whole chain sits relative to the other. Pick a residue in the first chain and treat it as the frame of reference; ask how well AlphaFold thinks it has placed every residue of the second chain; average those answers. Do that for every residue in turn, and keep the best one. Then repeat with the chains swapped and keep the better of the two.

**The formula.** `nx` and `ny` are the two chain lengths.

```
d0chn        = d0(nx + ny)                            # the whole complex sets the scale
ipTM_i(x→y)  = mean over ALL j in chain y of ptm(PAE[i, j], d0chn)
ipTM(x→y)    = max over i in chain x
ipTM_d0chn   = max( ipTM(x→y), ipTM(y→x) )
```

There is **no PAE cutoff**: every inter-chain cell contributes, however bad. That is the whole point of this score, and the whole problem with it.

**This is not AlphaFold's own ipTM.** AlphaFold2 does not emit a pairwise ipTM for a multi-chain complex and AlphaFold3 emits only the symmetric maximum, so Dunbrack recomputes an ipTM-like value from the PAE matrix and calls it `ipTM_d0chn`, "where d0chn indicates that d0 is calculated from the chain lengths" (2025, p. 21). It is close to AlphaFold's number but not identical to it: on his three-chain worked example the two agree to about 0.02–0.08 (0.443, 0.429, 0.752 against AlphaFold's 0.46, 0.51, 0.77), and he attributes the gap to "using the PAE values in the pairwise pTM matrix ... instead of the expectation value over the probability distribution of PAE" (p. 21). AlphaFold sums over 64 aligned-error bins; this reconstruction substitutes the single expected PAE. AFDB does not publish AlphaFold's own ipTM on these endpoints, so if you compare this number with an ipTM printed by AlphaFold itself, expect a small difference and do not read it as a bug.

**The thinking behind it, and why it is here.** ipTM_d0chn is included as the **control**, not as a recommendation. Dunbrack's paper is an argument against reading it, and its title says so: *What's wrong with AlphaFold's ipTM score and how to fix it*. Two artefacts, both traced to the equations above (pp. 7–8):

- **Add non-interacting sequence to one chain and the score goes up.** `d0` is computed from the summed chain lengths, so padding one chain enlarges `d0`, which makes every PAE value score better. "This causes an artifact in the calculation of ipTM, artificially raising the score when the user adds sequence to one of the chains" (p. 7).
- **Add non-interacting sequence to both chains and the score goes down.** Now every residue also acquires a crowd of terrible inter-chain PAE values which are averaged in: "Any mobile domains that do not interact also lower the score, because they also contribute low pTM_ij from low PAE_ij" (p. 8).

His worked case is KRAS with the RAS-binding domain of RAF1 (p. 8). On the trimmed domains ipTM is 0.85. Add 120 disordered residues to one chain and it rises to 0.90; add them to both and it falls to 0.56. **The predicted interface is the same structure in all three.** This matters directly here, because AFDB predictions are run on full-length UniProt sequences, disorder and accessory domains included.

**How to read the number.** Green ≥ 0.70, amber ≥ 0.30, red below. Both edges are **DERIVED**, from Dunbrack's benchmark of 40 true and 70 false dimers (p. 14): "In both panels, there is overlap in the density between values of ipTM or ipSAE from 0.3 to 0.7 for the true dimers ... and false dimers." Green means above the overlap; red means below it; **amber means inside it, where the score genuinely cannot tell a true dimer from a decoy.** The right-hand panel he is describing is ipTM computed from the PAE matrix, which is this exact quantity, so the band is measured on the number we compute rather than transferred from a relative.

A low ipTM_d0chn on a long protein is weak evidence of anything. Compare it with ipSAE_d0chn in 4.2: those two differ only in whether the PAE cutoff is applied, so the gap between them is a direct readout of how much dilution is going on.

In [ ]:
# 4.1 ipTM_d0chn. No PAE cutoff: every inter-chain cell contributes.
# The pair is ordered, and every "x -> y" label in Section 4 reads against it.
print(f'Ordered pair: {short_x} → {short_y}')
print(f'  {short_x} = {full_x}')
print(f'  {short_y} = {full_y}')

res_iptm = ciu.compute_iptm_d0chn(pair)
_band_iptm = ciu.traffic_light(res_iptm.score, 'iptm_d0chn')

print(f'\nipTM_d0chn : {res_iptm.score:.4f}   [{_band_iptm[1]}]')
print(f'  d0chn                 : {res_iptm.d0:.4f} '
      f'(from n0chn = {res_iptm.n0} = {nx} + {ny} residues)')
# The reported value is one residue's number, so name that residue.
print(f'  peak residue, {short_x} → {short_y}: '
      f'{chains[chain_x].res_ids[res_iptm.forward.argmax_index]} '
      f'(index {res_iptm.forward.argmax_index})')
print(f'  peak residue, {short_y} → {short_x}: '
      f'{chains[chain_y].res_ids[res_iptm.reverse.argmax_index]} '
      f'(index {res_iptm.reverse.argmax_index})')
print(f'  threshold             : green ≥ {ciu.THRESHOLDS["iptm_d0chn"].green}, '
      f'amber ≥ {ciu.THRESHOLDS["iptm_d0chn"].amber} '
      f'({ciu.THRESHOLDS["iptm_d0chn"].green_provenance})')

### 4.2 ipSAE — the same question, asked only where AlphaFold is confident

**What it measures.** The same average as ipTM_d0chn, with two changes: pairs where AlphaFold has no idea are **dropped instead of averaged in**, and the normalisation is rescaled to how many pairs survived. Dunbrack names it ipSAE, for "interaction prediction score from aligned errors" (2025, p. 3).

**The formula.** One mask, three normalisations.

```
valid[i, j]  = PAE[i, j] < 10             # strict; AFDB's production cutoff
ipSAE_i(x→y) = mean over valid j only of ptm(PAE[i, j], d0)
ipSAE(x→y)   = max over i          ipSAE = max( ipSAE(x→y), ipSAE(y→x) )
```

The three variants differ **only in the `L` that produces `d0`**, and in nothing else:

| variant | `L` is | so `d0` is | meaning |
|---|---|---|---|
| **d0res** | that residue's own count of sub-cutoff partners | different for every residue | the real ipSAE |
| **d0chn** | `nx + ny` | one value for the pair | ipTM's normalisation, with the cutoff applied |
| **d0dom** | residues of either chain with at least one sub-cutoff inter-chain pair | one value per direction | the interacting part of the complex |

Because `d0` never decreases with `L`, and `ptm` never decreases with `d0`, and `n0res_i ≤ n0dom ≤ nx + ny` always, the ordering

**`ipSAE_d0chn ≥ ipSAE_d0dom ≥ ipSAE_d0res`**

holds for every model. It is a theorem, not something we observed. Do not read three green lights as three confirmations.

**The thinking behind it.** Dunbrack lists three modifications to ipTM in the abstract (p. 2): "1) including only residue pairs in the ipTM metric that have good predicted aligned error (PAE) scores; 2) by adjusting the d0 parameter ... to include only the number of residues with good interchain PAEs to the aligned residue; and 3) using the PAE value itself and not the probability distributions". The third is a practical concession, so that the score can be computed from the JSON files AlphaFold already writes. **The first two are the science, and they only work together.**

Why the cutoff alone is not enough is stated plainly on p. 9: "a small number of interchain residue pairs with spuriously good PAE_ij and consequently good pTM_ij values may produce an unrealistically ipTM if d0 is not adjusted." A handful of lucky cells, divided by a `d0` sized for the entire complex, looks like a real interface.

His demonstration is a pair of proteins that do not interact, RAF1 and RIPK1 (pp. 11, 14). AlphaFold's per-residue ipTM peaks at 0.290. Apply the PAE cutoff but keep the old chain-length `d0`, and the score goes **up**, to 0.459, because you have thrown away all the bad pairs and kept the forgiving scale. Apply the cutoff *and* rescale `d0` to the 75 residues that survived it, and the score collapses to **0.044**, "indicating that the proteins are not likely to interact". On a genuine complex the two fixes push the other way: full-length RAF1 with KSR1 scores 0.41 by AlphaFold's ipTM and 0.73 by ipSAE (p. 11).

**Why three variants exist.** Dunbrack computes `d0chn` and `d0dom` "for comparison purposes" (p. 21); ipSAE proper is `d0res`. They are diagnostics for the same model, and the spread between them is the useful part. `d0chn` is what you get if you apply the cutoff but keep ipTM's normalisation, so **`ipSAE_d0chn` minus `ipSAE_d0res` measures how small the confident interface is relative to the two chains**. A wide gap says most of the complex is not participating.

**Why the cutoff is 10 Å.** Dunbrack scanned it and concluded "Cutoffs of 10 or 15 Å may be most suitable" (p. 14). AFDB's production pipeline used 10, which is what this notebook uses, so the AFDB thresholds below apply to these numbers directly.

**How to read the number.** The **metric is Dunbrack's**. The **thresholds are AlphaFold DB's**, from Han, Tsenkov, Venanzi et al. 2026, and they are published for `ipSAE_d0res` specifically, the version AFDB calls `ipSAEmax` (p. 12):

| band | ipSAE_d0res | what it means |
|---|---|---|
| **VERY HIGH-CONFIDENCE** | ≥ 0.80 | the interface is modelled with high accuracy; suitable for work that depends on interface detail, such as characterising a binding site or interpreting an interface mutation |
| **CONFIDENT** | 0.70 – 0.80 | a generally correct quaternary arrangement and contact pattern |
| **LOW-CONFIDENCE** | 0.60 – 0.70 | the protein may well form a dimer, but the precise interface geometry may not be accurate |
| **BELOW AFDB THRESHOLD** | < 0.60 | very low interface confidence. Note that **the individual chains may still be accurately modelled** (high pLDDT) even when the dimer interface is unreliable |

The 0.60 edge is also the ipSAE half of AFDB's release criterion, which the notebook evaluates as a PASS/FAIL badge in Section 7. The same edges are transferred to `d0chn` and `d0dom` so the three can be compared on one scale, but **only the numbers transfer, not the published band names**, and by the ordering theorem a shared threshold is strictly most permissive for `d0chn` and strictest for `d0res`. **Quote `ipSAE_d0res`.**

**0.6 is conservative, not optimal.** AFDB adopted it as a community-established cutoff by citation rather than deriving it, and validated it: precision 0.924 at a false-positive rate of 0.043 on their homodimer benchmark, 0.958 at 0.004 on the heterodimer benchmark, "supporting its use as a quality filter that prioritises precision over recall given the scale of the release" (2026, p. 5). Their own MCC-optimal cutoffs are far lower, **0.104** for homodimers and **0.520** for heterodimers (Supplementary Figs. 1–2, p. 24). A model below 0.6 has not been shown to be wrong; it has been declined for high-confidence release, and it stays downloadable with its scores.

**Three caveats that come with the bands.**

- **They are not statistically optimised on homodimers.** The AFDB technical note describing them says they are "evidence-informed rather than statistically optimised on a dedicated homodimer validation set", and that ipSAE "is not experimentally calibrated for homodimers specifically": Dunbrack's benchmark used heterodimers (note §7).
- **Do not over-interpret a boundary.** Scores vary with random seed and hardware, so "scores near tier boundaries (e.g., 0.59 vs 0.61) should not be over-interpreted; users should consider the score as a continuous measure, not a rigid classification" (note §7).
- **All-helical interfaces may score high for the wrong reason.** Alpha-helical proteins tend to achieve higher ipSAE than beta-sheet or mixed folds, "likely due to greater fold stability and lower structural ambiguity" (note §7).

**One more thing ipSAE does not tell you.** It is a statement about AlphaFold's confidence in an interface geometry, not about binding: "A high ipSAE indicates that AlphaFold is confident in the predicted interface geometry, but it does not directly predict whether the interaction occurs in vivo or its strength" (note §7).

In [ ]:
# 4.2 ipSAE. One mask (PAE < PAE_CUTOFF), three normalisations. The module
# computes all three in one pass because they share that mask.
res_ipsae = ciu.compute_ipsae(pair, pae_cutoff=PAE_CUTOFF)

print(f'PAE cutoff: {res_ipsae.pae_cutoff:.0f} Å (strict <)\n')
for _key, _variant in res_ipsae.variants.items():
    _colour, _label = ciu.traffic_light(_variant.score, _key)
    print(f'  {ciu.SCORE_DISPLAY_NAMES[_key]:12s}: {_variant.score:.4f}   [{_label}]')

# The d0 values are the point of the exercise: same PAE cells, three scales.
print(f'\n  d0chn : {res_ipsae.d0chn_value:.4f}  (n0chn = {res_ipsae.n0chn})')
print(f'  d0dom : {res_ipsae.d0dom.d0:.4f}  (n0dom = {res_ipsae.d0dom.n0}, '
      f'of the direction that supplied the reported value)')
print(f'  d0res : {res_ipsae.d0res.d0:.4f}  (n0res = {res_ipsae.d0res.n0}, '
      f'of the peak residue alone)')

# d0chn >= d0dom >= d0res is a theorem, not a measurement, so it is checked
# here rather than left for the reader to notice. It is reported rather than
# asserted: a violation would mean the computation is wrong, not the model, and
# that is worth seeing next to the numbers rather than as a traceback.
_ordered = (res_ipsae.d0chn.score >= res_ipsae.d0dom.score - 1e-6
            and res_ipsae.d0dom.score >= res_ipsae.d0res.score - 1e-6)
print(f'\n  ordering d0chn ≥ d0dom ≥ d0res : '
      f'{"holds" if _ordered else "VIOLATED — this should be impossible"}')
print(f'  d0chn - d0res spread          : '
      f'{res_ipsae.d0chn.score - res_ipsae.d0res.score:+.4f}')
print('  A wide spread means the confidently predicted interface is small')
print('  relative to the two chains. Quote ipSAE_d0res.')

### 4.3 pDockQ — how big the interface is, and how well-resolved

**What it measures.** A guess at DockQ, the standard score for "how close is this docked model to the real structure", made without ever seeing the real structure. It uses **no PAE at all**. Just two things: how many residue-to-residue contacts the interface has, and how confident AlphaFold was about those residues individually.

**The formula.** A contact is a CB–CB pair within 8 Å (CA for glycine, which has no CB).

```
npairs     = number of contact PAIRS across the interface
mean_pLDDT = mean pLDDT over the UNION of interface residues from both chains
x          = mean_pLDDT × log10(npairs)
pDockQ     = 0.724 / (1 + exp(-0.052 × (x - 152.611))) + 0.018
```

Two traps worth naming. `npairs` counts **pairs**, not residues, and it is several times larger than the interface residue count. And `mean_pLDDT` is an unweighted mean over each interface residue counted **once**, so a residue with forty contacts does not count forty times.

pDockQ is **symmetric**: swapping the chains changes neither the pair count nor the residue set, so unlike every other score here it has only one value, not two.

**The thinking behind it.** Bryant, Pozzati and Elofsson set out the problem in one sentence (2022, p. 3): "It is not only essential to obtain improved predictions, but also to be able to discriminate between acceptable and non-acceptable ones."

The obvious candidate, average pLDDT over the whole complex, does not work, and they say why: it achieves an AUC of only 0.66, "suggesting that both single chains in a complex are often predicted very well, while their relative orientation may still be incorrect" (p. 3). pLDDT is a local, within-chain measure; a perfectly folded pair of chains can be docked completely wrong.

So they tested five interface-level signals and combined the two best (p. 3): "The total number of interactions between Cβs and the number of residues in the interface can separate the correct/incorrect models with an AUC of 0.92 and 0.91 respectively, while the average interface plDDT results in an AUC of 0.88. However, pLDDT results in higher TPRs at lower FPRs; therefore, we multiply the plDDT with the logarithm of the interface contacts resulting in an AUC of 0.95." That is the actual reason for the product: contact count is the better discriminator overall, but interface pLDDT is the better one in the high-specificity regime where you want to operate, and multiplying keeps both.

Their reading of the contact-count term is that size is a proxy for how determined the answer is: "Assuming that all residues in an interface contribute to the interaction energy could explain why larger interfaces are more likely to be correctly predicted" (p. 7).

The sigmoid is not a probability. It is a curve fitted with SciPy's `curve_fit` **directly to observed DockQ scores** on a 1481-model test set, yielding "L = 0.724, x0 = 152.611, k = 0.052 and b = 0.018" (p. 10). So a pDockQ of 0.23 is a *predicted DockQ* of 0.23, which is what makes the threshold below transfer.

**How to read the number.** Green ≥ 0.23, amber ≥ 0.12, both **DERIVED**.

0.23 is the CAPRI acceptability boundary for DockQ itself, not a pDockQ recommendation: Bryant uses it as the definition of success ("acceptable quality (DockQ ≥ 0.23)", p. 1) but never writes "use pDockQ ≥ 0.23". The single inferential step is the one above, that pDockQ is fitted to predict DockQ. The amber of 0.12 comes from the score's own stated accuracy: pDockQ predicts DockQ "with an overall average error of 0.11 on the test set" (p. 4), so a model at 0.23 − 0.11 sits one average error below acceptable and could still be an acceptable model that pDockQ under-scored.

The sigmoid is bounded on [0.018, 0.742], so 0.23 is not far above the floor and the score cannot reach 1.0. Note also that `ipsae.py` returns exactly **0.0** for an interface with no contacts, rather than the 0.018 limit.

**A green pDockQ is not confirmation on its own.** Zhu et al. tested it outside the heterodimer regime it was fitted on and found it over-optimistic: "More than 10% of the chains in all these sets have pDockQ > 0.5 and DockQ_i < 0.23" (2023, p. 5). That is the reason 4.4 exists.

In [ ]:
# 4.3 pDockQ. No PAE anywhere in this one: contacts and pLDDT only.
res_pdockq = ciu.compute_pdockq(contacts, plddt_x, plddt_y)
_band_pdockq = ciu.traffic_light(res_pdockq.score, 'pdockq')

print(f'pDockQ : {res_pdockq.score:.4f}   [{_band_pdockq[1]}]')
print(f'  contact pairs (npairs) : {res_pdockq.n_contact_pairs}   '
      f'(CB-CB ≤ {res_pdockq.dist_cutoff:.0f} Å)')
print(f'  interface residues     : {res_pdockq.n_interface_residues}   '
      f'(counted once each; reported by ipsae.py but not used in the score)')
print(f'  mean interface pLDDT   : {res_pdockq.mean_plddt:.2f}')
print(f'  x = mean_pLDDT × log10(npairs) : {res_pdockq.x:.4f}')
print(f'  symmetric              : {res_pdockq.symmetric} '
      f'(checked by computing both orientations, not assumed)')
print(f'  threshold              : green ≥ {ciu.THRESHOLDS["pdockq"].green}, '
      f'amber ≥ {ciu.THRESHOLDS["pdockq"].amber} '
      f'({ciu.THRESHOLDS["pdockq"].green_provenance}); '
      f'sigmoid bounded on [0.018, 0.742]')

### 4.4 pDockQ2 — the same idea, with PAE at the contacts

**What it measures.** pDockQ, plus the question pDockQ never asks: *given that these two residues are touching, does AlphaFold actually believe they are positioned correctly relative to each other?* It replaces the contact-count term with the mean confidence at the contacts.

**The formula.** Same contact set as pDockQ, same mean pLDDT.

```
mean_ptm   = mean over contact pairs of ptm(PAE[i, j], d0 = 10)      # fixed d0
mean_pLDDT = mean pLDDT over the union of interface residues
x          = mean_pLDDT × mean_ptm
pDockQ2    = 1.31 / (1 + exp(-0.075 × (x - 84.733))) + 0.005
```

`d0` is **fixed at 10 Å** here and is not derived from any length; Zhu calls it "an optimized parameter" (2023, p. 3). Unlike pDockQ, pDockQ2 **is directional**: `mean_ptm` reads only one direction's PAE block, so the reported value is the larger of the two.

**The thinking behind it.** Zhu et al. wanted a per-interface number, because AlphaFold's own scores are whole-complex averages: "Both these scores estimate the average quality of the complex (or all interfaces of the complex) ... However ... it is sometimes desirable to estimate each interface's quality within a multichain complex" (p. 3). And they observed the failure directly: "Many models have a low min DockQ_i while a high pTM and ipTM" (p. 5).

Then they turned the same test on pDockQ and it failed too, in a specific and instructive way. All of this is on p. 5:

> "pDockQ does not utilize the predicted average errors (PAEs) but only considers the size of the interface and the predicted quality (pLDDT) of residues in the interface. **Therefore, it does not work if a method generates models with large, highly confident incorrect interfaces.** To correctly classify such models as wrong, it is necessary to consider the PAE."

That is the entire argument for pDockQ2 in three sentences. A big interface built from residues each of which AlphaFold is individually sure about will score well on pDockQ no matter how wrongly the two chains are placed against each other, because nothing in pDockQ measures the *relative* placement. PAE does. They also identify why pDockQ mis-fires here: it "was developed to predict DockQ on a dataset of heteromeric dimer models created with FoldDock", and on homomers, on multimers, and on AlphaFold-Multimer models generally it "sometimes gives high scores to incorrect models" (p. 5).

The sigmoid is fitted the same way as pDockQ's, to the same target: "As in pDockQ, we fit a sigmoid curve (Equation 1) (by the scipy package) to the actual DockQ_i values, yielding the coefficients L = 1.31, x0 = 84.733, k = 0.075, and b = 0.005" (p. 3).

**Zhu is candid about pDockQ2's own failure modes**, and both are worth knowing here. High pDockQ2 with a wrong pose still happens: among 72 extreme cases, "usually individual chains are predicted quite accurately, but the docking positions are wrong compared with the native structures. However, the high pDockQ2 values indicate that AlphaFold-Multimer is quite confident regarding the interface residues" (p. 6). And some apparent false positives are the reference structure's fault rather than the model's, which is particularly relevant to homodimers: in one case "the dimer was cut out of a crystallization structure of pantoate kinase, which has 2-fold homodimers ... In other words, the biological assembly from PDB might not represent the only biological assembly" (p. 6).

**How to read the number.** Green ≥ 0.23 (**PUBLISHED**, and independently double-sourced), amber ≥ 0.10 (**a judgement call**).

Zhu actually applies 0.23 as an operating cutoff to select biological conclusions: "9 had pTM > 0.5 and min pDockQ2 > 0.23" (p. 6). AFDB arrives at the same number independently, as the second half of its release criterion: "pDockQ2max ≥ 0.23, corresponding to the DockQ 'acceptable' quality boundary" (Han, Tsenkov, Venanzi et al. 2026, p. 5). Its job in that rule is to catch clash-prone predictions that ipSAE alone lets through, so the corroboration is genuine rather than a restatement. AFDB's own MCC-optimal cutoff for pDockQ2 is 0.013 on the homodimer benchmark (Supplementary Fig. 1, p. 24), so 0.23 is conservative here too.

The amber of 0.10 has no publication behind it. Zhu uses `pDockQ2 < 0.1` as a bucket boundary when analysing outliers (p. 6), which is not a recommendation, and the notebook does not present it as one.

**Never call pDockQ2 a probability.** Its sigmoid is bounded on [0.005, **1.315**] and can exceed 1.0, which no DockQ-scaled quantity should.

In [ ]:
# 4.4 pDockQ2. Same contacts and same mean pLDDT as 4.3; the contact-count
# term is replaced by mean ptm(PAE, d0=10) over those contacts, which is read
# from one direction's PAE block and is therefore directional (R007).
res_pdockq2 = ciu.compute_pdockq2(contacts, pair, plddt_x, plddt_y)
_pdockq2_dir = res_pdockq2.winning_direction
_band_pdockq2 = ciu.traffic_light(res_pdockq2.score, 'pdockq2')

print(f'pDockQ2 : {res_pdockq2.score:.4f}   [{_band_pdockq2[1]}]')
print(f'  reported direction  : {_pdockq2_dir.chain_row} → {_pdockq2_dir.chain_col} '
      f'(the larger of the two)')
print(f'  {short_x} → {short_y} : {res_pdockq2.forward_score:.4f}      '
      f'{short_y} → {short_x} : {res_pdockq2.reverse_score:.4f}      '
      f'|Δ| {res_pdockq2.delta:.4f}')
print(f'  contact pairs       : {_pdockq2_dir.n_contact_pairs}')
print(f'  mean_ptm (d0 = 10)  : {_pdockq2_dir.mean_ptm:.4f}   '
      f'← the ingredient pDockQ does not have')
print(f'  mean interface pLDDT: {_pdockq2_dir.mean_plddt:.2f}')
print(f'  x = mean_pLDDT × mean_ptm : {_pdockq2_dir.x:.4f}')
print(f'  threshold           : green ≥ {ciu.THRESHOLDS["pdockq2"].green} '
      f'({ciu.THRESHOLDS["pdockq2"].green_provenance}), '
      f'amber ≥ {ciu.THRESHOLDS["pdockq2"].amber} '
      f'({ciu.THRESHOLDS["pdockq2"].amber_provenance}); '
      f'bounded on [0.005, 1.315], not a probability')

### 4.5 LIS — is there a confident patch at all?

**What it measures.** How much of the inter-chain PAE block is confident, and how confident. It answers *do these two proteins interact?* rather than *is this pose right?*, and those are different questions with different right answers.

**The formula.** No `d0`, no TM-score transform, no structure. Just the PAE block.

```
LIA        = the inter-chain cells with PAE < 12          # the "local interaction area"
LIS(x→y)   = mean over LIA cells of (12 - PAE) / 12
LIS        = ( LIS(x→y) + LIS(y→x) ) / 2                  # the MEAN, not the max
```

Two details that are easy to get wrong. The cutoff is **12 Å and is not the ipSAE cutoff**; it is hardcoded in `ipsae.py` and changing it invalidates the threshold below. And the two directions are combined by **averaging**, which is the one score here that is not a maximum. The paper's own worked example does the same: an A:B score of 0.457 and a B:A score of 0.309 combine as `(0.457 + 0.309) / 2 = 0.383` (Kim 2024, Fig. 2A, p. 25).

*One honest caveat about the formula.* Kim et al. describe the transform only as "inversely mapping PAE values within the LIA" to a 0-to-1 scale (p. 5). The specific linear form `(12 - PAE) / 12` comes from their reference implementation, which `ipsae.py` follows, not from the text of the paper.

**The thinking behind it.** Kim et al. start from a limitation of ipTM that is different from Dunbrack's. Their objection is not that ipTM is diluted; it is that ipTM is measuring the wrong thing for their purpose (p. 4):

> "High ipTM scores are often associated with stable, enduring PPIs ... Nonetheless, a focus on structural accuracy might not provide a complete representation of the various types of PPIs that take place within physiological environments, **where interactions are often local and transient.**"

Many real interactions run through a short linear motif or a small patch on a flexible region, and "flexible regions like IDRs often decrease overall protein structure accuracy" (p. 3), so a whole-complex accuracy score buries them.

The construction came from looking at PAE maps (p. 5): "we noticed that a few PPIs with low ipTM scores nevertheless had significant areas of blue color on PAE maps, indicative of regions with lower alignment error that we interpret to be probable sites of interaction". So they kept the blue and threw away the rest, "while ignoring areas with higher PAE values (in red), which are unlikely to correspond to interaction zones", then inverted and averaged what remained. The cutoff of 12 was not chosen by hand: they scanned PAE cutoffs from 1 to 30 and took the one that maximised ROC AUC (pp. 5–6).

The result behaves as intended. LIS separates positive reference sets from negative controls better than ipTM does (p. 6), it recovers interactions ipTM misses ("identifying many PPIs from PRS with low ipTM scores but relatively high LIS values ... This indicates that there are interactions that ipTM-based evaluations could miss", p. 6), and it is largely decoupled from how well the individual chains were folded: "Among all the metrics, LIS exhibited the lowest correlation with pLDDT ... and pTM ... This suggests that LIS has the capability to identify potential PPIs regardless of structural accuracy" (p. 7).

**How to read the number.** Green ≥ 0.21 (**PUBLISHED**), amber ≥ 0.10 (**a judgement call**).

0.21 is Kim's ROC/Youden-optimal cutoff for the rank-1 model, which is what an AFDB entry gives you: "The selected PPIs exhibit best LIS values exceeding the optimal threshold (0.21)" (p. 30), with thresholds "established based on the highest Youden's Index" (p. 15). For calibration: at these thresholds "over 70% of yeast positive PPIs and ELM-annotated PPIs were predicted to be direct interactions. Approximately 5% of yeast negative control sets had values that exceeded these thresholds" (p. 7). So green LIS is roughly a 5%-false-positive operating point on their reference sets.

The amber of 0.10 is ours. **Kim publishes exactly one cutoff per metric and no gradation below it**, which a Youden-optimal threshold is by construction; there is no second tier to quote. LIS has no sigmoid floor, so 0.0 means literally no inter-chain PAE below 12, and the 0.10–0.21 range does mean "there is a confident patch, but a weak or small one".

**Green LIS with red pDockQ is a legitimate combination**, not a contradiction: LIS says the pair looks like it interacts, pDockQ says the pose is not well determined. One known failure mode: a very high LIS computed over a very small patch. Kim hit this and added an area filter for it, since "some PPIs with very high LIS had very low LIA values, which might be indicative of false positive discovery" (p. 8). The printed cell below reports the number of contributing cells alongside the score for exactly that reason.

In [ ]:
# 4.5 LIS. The cutoff is 12 Å and is deliberately not PAE_CUTOFF: Kim 2024
# chose 12 by ROC, and the published 0.21 threshold is only valid there.
res_lis = ciu.compute_lis(pair, lis_cutoff=LIS_CUTOFF)
_band_lis = ciu.traffic_light(res_lis.score, 'lis')

print(f'LIS : {res_lis.score:.4f}   [{_band_lis[1]}]   '
      f'(the MEAN of the two directions, not the max)')
print(f'  {short_x} → {short_y} : {res_lis.forward_score:.4f}      '
      f'{short_y} → {short_x} : {res_lis.reverse_score:.4f}      '
      f'|Δ| {res_lis.delta:.4f}')

# The size of the local interaction area, alongside the score: Kim 2024 p. 8
# reports high LIS over a very small LIA as a false-positive signature.
_names = {chain_x: short_x, chain_y: short_y}
for _d in (res_lis.forward, res_lis.reverse):
    print(f'  LIA, {_names[_d.chain_row]} → {_names[_d.chain_col]} : '
          f'{_d.n_valid_pairs} of {_d.n_pairs} inter-chain cells below '
          f'{res_lis.lis_cutoff:.0f} Å ({100 * _d.fraction_valid:.1f}%)')
print(f'  threshold        : green ≥ {ciu.THRESHOLDS["lis"].green} '
      f'({ciu.THRESHOLDS["lis"].green_provenance}), '
      f'amber ≥ {ciu.THRESHOLDS["lis"].amber} '
      f'({ciu.THRESHOLDS["lis"].amber_provenance})')

### 4.6 All seven values together

The cell below collects the scores computed in 4.1 to 4.5 into one dictionary, keyed the way `ciu.THRESHOLDS` is keyed, so the summary table, the agreement matrix and every traffic light in Section 7 read one set of names. **Nothing is recomputed here.**

**Seven numbers are not seven votes.** Three of them are ipSAE under three normalisations and are ordered by a theorem, so they cannot disagree in the direction that would make them independent. ipTM_d0chn and ipSAE_d0chn differ only by the PAE cutoff. Genuinely separate lines of evidence: **ipSAE_d0res** (confident-pair geometry), **pDockQ** (interface size and local resolution, no PAE), **pDockQ2** (PAE at the contacts), and **LIS** (is there a confident patch at all). Section 7 compares those five.

**If you know one of these scales and not the others**, the AFDB technical note lines up the tiers. These are analogous quality bands from independent analyses of the same problem, not conversions:

| Quality tier | ipSAE | ipTM (AlphaFold's own) | DockQ |
|---|---|---|---|
| Very high | ≥ 0.8 | > 0.8 | > 0.80 (High) |
| Confident | 0.7 – 0.8 | 0.6 – 0.8 (upper) | 0.49 – 0.80 (Medium) |
| Low confidence | 0.6 – 0.7 | 0.6 – 0.8 (lower) | 0.23 – 0.49 (Acceptable) |
| Very low | < 0.6 | < 0.6 | < 0.23 (Incorrect) |

The ipTM grey zone is split across two ipSAE tiers deliberately, because ipSAE is more discriminative: in Dunbrack's benchmark, values above 0.7 overlapped far less with false dimers than values in 0.6–0.7. DockQ is a ground-truth measure computed against an experimental structure, while ipSAE is a prediction, so that column is the loosest of the three correspondences.

**One caveat that applies to homodimers specifically, and to nothing else in this notebook.** AlphaFold has a markedly higher false-positive rate on homodimers than on heterodimers. On the benchmark the AFDB note cites, **the true-positive rate at a 1% false-positive rate falls from 63% for heterodimers to 18% for homodimers** (note §3.9, citing Elofsson et al. 2025). The reason is biological rather than numerical: many proteins that do not homodimerise in vivo have homologs that do, so AlphaFold predicts a plausible but non-physiological interface with every appearance of confidence. If both chains here are the same protein, read every score below with that in mind, and treat known biology, conservation and the PAE map as evidence in their own right rather than as decoration.

In [ ]:
# 4.6. Collection only. Every value below was computed in 4.1 to 4.5 and is
# read back here, so this cell cannot change a number.
# Keyed by `ciu.THRESHOLDS` key, so the summary table, the agreement matrix and
# every traffic light below read one set of names.
scores = {
    'ipsae_d0res': res_ipsae.d0res.score,
    'ipsae_d0chn': res_ipsae.d0chn.score,
    'ipsae_d0dom': res_ipsae.d0dom.score,
    'iptm_d0chn':  res_iptm.score,
    'pdockq':      res_pdockq.score,
    'pdockq2':     res_pdockq2.score,
    'lis':         res_lis.score,
}

print('\n── Score Results ──────────────────────────────')
for _name, _value in scores.items():
    print(f'  {ciu.SCORE_DISPLAY_NAMES[_name]:18s}: {_value:.4f}')

#### Which direction is the score measured in?

The PAE matrix is **not symmetric**. `PAE[i, j]` is AlphaFold's error estimate for residue `i` when residue `j` is used as the alignment frame, and asking the same question with the roles swapped gives a different answer. Every inter-chain score is therefore computed **twice** — once reading the first chain of the pair against the second, and once the other way round — and the single number reported for the complex is a *combination* of those two measurements, never one of them:

| Score | Combined as | Why |
|---|---|---|
| ipTM_d0chn, ipSAE_d0res, ipSAE_d0chn, ipSAE_d0dom, pDockQ2 | **max** of the two directions | this is what `ipsae.py` computes, and it is the value AlphaFold DB publishes for the entry |
| LIS | **mean** of the two directions | LIS is the one value `ipsae.py` averages rather than maximises |
| pDockQ | **not directional** | swapping the chains leaves the contact-pair count and the interface residue set unchanged, so there is only ever one measurement to report |

**This is worth showing rather than collapsing.** On the heterodimer `AF-0000000211034637`, ipSAE_d0res is **0.5555** measured one way and **0.7057** measured the other — a spread of 0.15, which straddles AlphaFold DB's 0.6 classification cutoff. AFDB publishes both per-direction values for that entry and they match the two numbers below to six decimal places. A reader who does not know the reported score is a maximum over directions will read 0.7057 as a property of the complex, when it is a property of one chain's view of the other.

**The headline number does not move.** The `Reported` column below is exactly what `ipsae.py` and AFDB report, and it is what Section 7's summary table and traffic lights use. `DIRECTION` in Section 1 changes only which direction is marked here and which panel the profile figure draws.

In [ ]:
# R008. `directional_deltas` reads the per-direction values the `compute_*`
# functions already kept, so nothing here recomputes a score — this cell cannot
# change a number, only show one that was previously collapsed out of sight.
# `DIRECTION` is resolved against the real chain ids, so a typo fails here with
# a message rather than silently showing the wrong direction.
_direction = ciu.resolve_direction(DIRECTION, chain_x, chain_y)

_deltas = ciu.directional_deltas(iptm_d0chn=res_iptm, ipsae=res_ipsae,
                                 pdockq=res_pdockq, pdockq2=res_pdockq2,
                                 lis=res_lis)

print(ciu.format_directional_report(_deltas, chain_x, chain_y,
                                    label_x=short_x, label_y=short_y,
                                    direction=_direction))

# n0dom is the mechanism, not a separate finding: the domain size that sets
# d0dom is itself counted per direction (R003), so the two d0dom columns above
# are not the same measurement under two names.
print(f'\n  n0dom  {short_x} → {short_y}: {res_ipsae.n0dom_xy}'
      f'   {short_y} → {short_x}: {res_ipsae.n0dom_yx}'
      f'   |Δ| {res_ipsae.n0dom_delta}')
print(f'  d0dom  {short_x} → {short_y}: {res_ipsae.d0dom_xy:.4f}'
      f'   {short_y} → {short_x}: {res_ipsae.d0dom_yx:.4f}')

#### Per-residue score profiles

Each panel below is **one direction** of the pair: the top panel plots residues of the first chain measured against the whole of the second, the bottom panel the reverse. (With `DIRECTION` set in Section 1, only that panel is drawn.)

**Why are only four lines plotted, and not all seven scores?** Because only these four have a genuine per-residue decomposition. Each of them is defined residue by residue — row `i` of the inter-chain PAE block is residue `i`'s own measurement against the entire partner chain — and the value reported for the direction is the **maximum over that profile**. The reported score is therefore literally one residue's number, which is why each series' peak is starred and labelled with its residue number and value. If you read the starred value off the `ipSAE_d0res` line, you are reading the score in Section 7's table.

The other three scores have nothing per-residue to draw:

| Score | What it pools over | Per-residue value? |
|---|---|---|
| **pDockQ** | the whole interface residue set — one mean pLDDT and one contact-pair count go into one sigmoid | no |
| **LIS** | every inter-chain PAE cell below 12 Å — a *pair*, not a residue; nothing in the formula is indexed by row | no |
| **pDockQ2** | every contact *pair*: `sigmoid(mean pLDDT × mean ptm)` over all of them | not one that belongs here — see below |

**pDockQ2 is the interesting near-miss.** The module does expose `mean_ptm_by_residue`, the mean of `ptm(PAE, d0 = 10)` over each residue's own contact pairs. It is a real per-residue quantity, but it is deliberately **not** on this plot: pDockQ2 is a sigmoid of a *pair-pooled* mean, so the score is not the maximum of that array, and putting it on an argmax-annotated figure would attach a meaningless peak to it. It is also undefined (`NaN`) for every residue with no contact — the great majority of the chain — so it would render as a broken line. Its natural home is Section 6, painted onto the structure itself, where the contact set it lives on is what you are looking at.

**How to read a panel.** The grey fill is pLDDT on the right-hand 0–100 axis; the amber band marks interface residues. A peak sitting on a low-pLDDT stretch, or well away from the amber band, is worth a second look. The three ipSAE variants differ only in the `L` that feeds `d0`, so `d0chn ≥ d0dom ≥ d0res` always: the vertical gap between those three lines is the price of the stricter normalisation, not a disagreement.

In [ ]:
# `direction` selects a panel; `res_ids_*` let the peak labels carry each
# residue's real number rather than its positional index (R062).
display(ciu.plot_residue_score_profiles(res_iptm, res_ipsae, contacts,
                                        plddt_x, plddt_y,
                                        label_x=short_x, label_y=short_y,
                                        direction=_direction,
                                        res_ids_x=chains[chain_x].res_ids,
                                        res_ids_y=chains[chain_y].res_ids))

---
## Section 5 — Per-Residue pLDDT at the Interface

**What is pLDDT?**
pLDDT (predicted Local Distance Difference Test) is AlphaFold's confidence score for each individual residue, ranging from 0 to 100. A score above 90 means AlphaFold is very confident about that residue's position within its local neighbourhood; below 50 means the residue is likely disordered.

**Why look at interface pLDDT separately?**
pDockQ scores depend on the pLDDT of interface residues. If the interface contains many low-pLDDT residues, pDockQ will be dragged down even if the structural contacts look physically reasonable. Comparing interface vs. non-interface pLDDT distributions helps us understand whether a low pDockQ reflects a *globally* uncertain protein or specifically uncertain *interface* residues.

**AlphaFold colour scheme:** dark blue (>90) → light blue (70–90) → yellow (50–70) → orange (<50).

In [ ]:
if_plddt = np.concatenate([plddt_x[contacts.mask_x], plddt_y[contacts.mask_y]])
ni_plddt = np.concatenate([plddt_x[~contacts.mask_x], plddt_y[~contacts.mask_y]])
n_low_if = int((if_plddt < 70).sum())

print(f'Interface pLDDT   mean={if_plddt.mean():.1f}, median={np.median(if_plddt):.1f}')
print(f'Non-interface     mean={ni_plddt.mean():.1f}, median={np.median(ni_plddt):.1f}')
print(f'Low-pLDDT (<70) interface residues: {n_low_if} / {len(if_plddt)} '
      f'({100 * n_low_if / max(len(if_plddt), 1):.1f}%)')

display(ciu.plot_plddt_distribution(contacts, plddt_x, plddt_y,
                                    label_x=short_x, label_y=short_y))


---
## Section 6 — 3D Structure Visualisation (MolViewSpec)

**What is MolViewSpec?**
MolViewSpec is a format for describing 3D molecular visualisations. We use it to colour the protein structure by different diagnostic criteria and then view it in **Mol***, the same interactive 3D viewer used by the PDBe and RCSB databases. Each view is interactive: drag to rotate, scroll to zoom, click a residue for its identity.

**All six views draw the same structure. Only the colouring rule changes.** Nothing here recomputes a score; every colour comes from a number an earlier section already produced, put back on the atoms it was measured from. Each view carries a legend built from the same constants the view is painted with, with the cutoffs actually in force written out, so a legend cannot quote a number the picture did not use.

**View 1 is drawn in Section 2, not here.** It is the only view coloured by chain identity rather than by a number, and it is the 3D counterpart of the contact map, so it belongs where the interface is defined. The numbering runs continuously across the two sections: a reference to "View 1" below points back to Section 2. What follows here is Views 2 to 6, every one of which paints a per-residue value onto the structure.

| # | View | Coloured by | Covers | Drawn in |
|---|---|---|---|---|
| 1 | Chain overview | chain identity | every residue | **Section 2** |
| 2 | pLDDT | AlphaFold's four published confidence bands | every residue | Section 6 |
| 3 | ipSAE d0res on the interface | each interface residue's own ipSAE_d0res | interface residues | Section 6 |
| 4 | ipSAE against contact | whether PAE and geometry agree | residues where either signal fires | Section 6 |
| 5 | pDockQ2 contact quality | confidence at each residue's own contacts | interface residues | Section 6 |
| 6 | pDockQ2 against ipSAE | whether the two per-residue signals agree | interface residues | Section 6 |

View 1 describes the geometry and View 2 describes the model. Views 3 to 6 describe the *scores*, and each has its own explanation below.

**If both chains here are the same protein, the homodimer caveat from section 4.6 applies to every view below.** A confident-looking, well-packed, deep-green interface is a statement about AlphaFold's confidence in an interface geometry. It is not evidence that the protein homodimerises in vivo.

In [ ]:
# Section 6 setup: pull the per-residue arrays the five views below are painted
# from. `_source` and `render_view` already exist: Section 2 resolved the
# structure once for View 1, and every view since reuses them (R030). Each view
# below is one cell, so the prose explaining a view sits directly above it (R070).

# Both scores are directional: `forward` measures chain x against chain y,
# `reverse` the other way, so each chain is painted with its own measurement
# rather than with its partner's (R007, R008).
_ipsae_x = res_ipsae.d0res.forward.values
_ipsae_y = res_ipsae.d0res.reverse.values
_ptm_x = res_pdockq2.forward.mean_ptm_by_residue
_ptm_y = res_pdockq2.reverse.mean_ptm_by_residue


if _source is not None:
    _contact_pae = ciu.contact_ptm_to_pae(ciu.MVS_CONTACT_PTM_THRESHOLD)
    print(f'Chains in these views: {full_x}; {full_y}')
    print(f'Contact, Views 4, 5, 6:     CB within {ciu.DIST_CUTOFF:.1f} A '
          '(CA for glycine)')
    print(f'Confident PAE, Views 4, 6:  ipSAE_d0res >= '
          f'{ciu.MVS_DISAGREEMENT_THRESHOLD:.2f} '
          "(THRESHOLDS['ipsae_d0res'].amber, AFDB's release edge)")
    print(f'Well-placed contacts, V6:   mean contact ptm >= '
          f'{ciu.MVS_CONTACT_PTM_THRESHOLD:.2f}, i.e. mean contact PAE better '
          f'than {_contact_pae:.0f} A')

### View 2 — pLDDT per residue

**What is drawn.** Every residue of both chains, interface or not.

**What the colours encode.** AlphaFold's own four published pLDDT bands, in AlphaFold's own colours: dark blue `> 90` (very high), light blue `70–90` (confident), yellow `50–70` (low), orange `< 50` (very low). The notebook uses AlphaFold's strict `>` ladder, so a residue at exactly 90.0 is *confident*, not *very high*.

**What to look for.** Orange and yellow stretches, which are usually disordered tails and long loops, and whether they overlap the interface side chains from View 1, in Section 2.

**What a problem looks like.** A low-pLDDT interface. pDockQ's input is `mean interface pLDDT × log10(number of contact pairs)`, so an interface built out of yellow residues cannot score well however many contacts it makes. Section 5 puts the same comparison on a distribution plot.

In [ ]:
render_view('plddt',
            lambda: ciu.build_plddt_view(_source, chains, contacts,
                                         plddt={chain_x: plddt_x,
                                                chain_y: plddt_y}),
            lambda: ciu.plddt_legend({short_x: plddt_x, short_y: plddt_y}))

### View 3 — ipSAE d0res on the interface

**What is drawn.** The whole complex as a mid-grey cartoon for context, and then every interface residue of **both** chains as ball-and-stick, coloured by its own per-residue ipSAE_d0res.

**What that value is, one residue at a time.** For a residue *i*, ipSAE_d0res is the average of `ptm(PAE[i, j], d0res_i)` over the partner-chain residues *j* whose inter-chain PAE is below 10 Å — and over no others. In words: *from this residue, how confidently does AlphaFold place the residues of the other chain that it has any opinion about at all?* The single number reported in section 4.2 is the **maximum** of this array over every residue and both directions. This view is that headline score taken apart and laid back on the structure.

#### Why `d0res`, and not `d0chn` or `d0dom`

Two reasons, pointing the same way.

**1. It is the variant AlphaFold DB actually publishes.** AFDB's `ipSAEmax` — the score its four confidence bands are defined on, and the ipSAE half of its release criterion — is the `d0res` variant (Han, Tsenkov, Venanzi et al. 2026, p. 12). The other two exist because Dunbrack computes them "for comparison purposes" (2025, p. 21); no publication states a cutoff for either, so the numbers in this notebook's traffic light are transferred to them rather than calibrated on them.

**2. It is the only variant whose `d0` varies along the chain.** This is what makes it the one worth painting.

| variant | `d0` derived from | so `d0` is |
|---|---|---|
| **d0res** | that residue's own count of sub-cutoff partners | **different for every residue** |
| d0chn | `n_x + n_y` | one value for the whole chain pair |
| d0dom | residues of either chain with any sub-cutoff inter-chain pair | one value per direction |

In `d0chn` and `d0dom` every residue is read through the *same* transform, so the only thing varying from residue to residue is that residue's own row of PAE values. In `d0res`, how many partners a residue is confident about changes the scale it is judged on: a residue confident about three partners is not allowed to score like a residue confident about ninety. That is a per-residue property, and it is the one a per-residue picture should be showing.

Painting `d0chn` instead would not reveal a different interface. The ordering that governs the three headline scores in section 4.2 holds residue by residue too — a residue's count of sub-cutoff partners is at most the domain count, which is at most `n_x + n_y`, and both `d0` and `ptm` increase with that length — so each residue's `d0chn` value sits at or above its `d0res` value. You would get a more forgiving rendering of the same rows.

#### What "low", "mid" and "high" actually mean

The ramp is continuous red-to-green over `0.00` to `1.00`, linear, with no steps in it. The legend beneath the view samples that same ramp at five points, which makes those swatches the ruler. Against AFDB's published bands:

| ipSAE_d0res | ramp colour | AFDB band |
|---|---|---|
| 1.00 | dark green `#006837` | very high-confidence |
| 0.80 | mid green `#66BD63` | **very high-confidence** starts here (≥ 0.80) |
| 0.70 | yellow-green `#A4D869` | **confident** starts here (0.70 – 0.80) |
| 0.60 | pale yellow-green `#D9EF8B` | **low-confidence** starts here (0.60 – 0.70) |
| 0.50 | pale yellow `#FEFEBD` | below the AFDB threshold |
| 0.25 | orange `#F88E52` | below the AFDB threshold |
| 0.00 | dark red `#A50026` | below the AFDB threshold |

Read practically:

- **Any green at all, from pale yellow-green upward, is at or above 0.60** — AFDB's release edge. Pale yellow-green is AFDB's own *low-confidence* band, clear green is *confident*, deep green is *very high-confidence*.
- **Pale yellow is about 0.50**, already below AFDB's threshold.
- **Orange and red are 0.30 and below.** That is the range Dunbrack's non-interacting control pairs occupy: 0.019, 0.012 and 0.000 for three pairs that do not interact (2025, Table 1, p. 16).

**Two things the colours will not do for you.** The ramp's midpoint sits at 0.50, which is not a band edge, so the four AFDB bands are *not* colour breaks — to name a residue's band exactly, compare it against the legend swatches rather than against your own sense of "greenish". And because the reported score is a maximum over residues, a view in which one residue is deep green and the rest are pale produces the same headline number as a view that is deep green throughout.

**What to look for.** Whether the green residues form one contiguous patch on each chain, and whether that patch coincides with the interface side chains in View 1, in Section 2.

**What a problem looks like.** A largely yellow or orange interface. If the headline ipSAE is high anyway, the score is resting on a small number of residues, and the per-residue profile figure at the end of Section 4 marks exactly which residue that is.

In [ ]:
# `build_interface_value_view` is deliberately score-agnostic: any per-residue
# array plus a colormap. View 5 below is the same call with a different array.
render_view('interface_value',
            lambda: ciu.build_interface_value_view(_source, chains, contacts,
                                                   _ipsae_x, _ipsae_y),
            lambda: ciu.value_ramp_legend('ipSAE d0res'))

### View 4 — ipSAE d0res against physical contact

**What is drawn.** The whole complex as a near-white cartoon, and on top of it only the residues for which at least one of two signals fires, coloured by *which*.

**The two signals, and where their numbers come from.**

- **Physical contact.** This residue's CB atom (CA for glycine) is within **8.0 Å** of a CB in the partner chain. Pure geometry, read off the coordinates: no PAE anywhere in it. Same contact definition pDockQ and pDockQ2 use (section 4.3).
- **Confident inter-chain PAE.** This residue's `ipSAE_d0res` is at or above **0.60**, read from `THRESHOLDS['ipsae_d0res'].amber` — AFDB's release edge, the same line Section 7's traffic light turns red below. It is not a number chosen for this figure.

The two are genuinely independent, which is the whole point of putting them in one picture.

**What the colours encode.**

| colour | category | meaning |
|---|---|---|
| bluish green `#009E73` | confirmed contact | in contact **and** confident. Both signals agree |
| blue `#0072B2` | predicted but not touching | confident, but no CB within 8.0 Å |
| dark red `#A02020` | touching but not trusted | in contact, but `ipSAE_d0res` below 0.60 |
| — | not drawn | neither signal fires. Most of the protein |

The legend counts each category per chain, so "how much of this interface is confirmed" is a number rather than an impression, and the last row accounts for the residues that were analysed and painted nothing.

**Blue is not automatically an error.** `ipSAE_d0res` averages over *every* partner residue with an inter-chain PAE below 10 Å, not only the ones a residue touches, so a residue can clear the cutoff without being in contact with anything. Being confident about where the other chain is does not require touching it.

**What to look for.** Whether the green residues form one contiguous patch matching the interface side chains in View 1, in Section 2, and how much blue surrounds it.

**What a problem looks like.**

- **A lot of dark red.** There is a physical interface that the PAE matrix does not support. This is the geometry behind a pDockQ that outruns ipSAE, and it is the failure Zhu et al. built pDockQ2 to catch: pDockQ "does not work if a method generates models with large, highly confident incorrect interfaces. To correctly classify such models as wrong, it is necessary to consider the PAE" (2023, p. 5).
- **Blue over a whole face with almost no green.** The PAE matrix is confident about an arrangement that the coordinates never realise as contacts at 8 Å.

In [ ]:
render_view('disagreement',
            lambda: ciu.build_disagreement_view(_source, chains, contacts,
                                                _ipsae_x, _ipsae_y),
            lambda: ciu.disagreement_legend(contacts, _ipsae_x, _ipsae_y,
                                            short_x, short_y))

### View 5 — pDockQ2 contact quality per interface residue

**What is drawn.** The same grey cartoon and the same interface residues as View 3, on the same red-to-green ramp — coloured by a **different measurement of the same residues**.

**What that value is, one residue at a time.** The mean of `ptm(PAE[i, j], d0 = 10)` over residue *i*'s **own contact pairs**: the partner-chain residues whose CB is within 8.0 Å. This is the quantity pDockQ2 pools away. Section 4.4's `mean_ptm` is this same average taken over every contact pair in the interface at once, so this view is pDockQ2's PAE term decomposed onto the structure.

Because `d0` is fixed at 10 Å here and never derived from a length, the colour converts straight back into Angstrom:

| mean PAE over this residue's contacts | value | ramp colour |
|---|---|---|
| 0 Å | 1.00 | dark green `#006837` |
| 5.8 Å | 0.75 | green `#84CA66` |
| 10 Å | 0.50 | pale yellow `#FEFEBD` |
| 17.3 Å | 0.25 | orange `#F88E52` |

10 Å is also ipSAE's `PAE_CUTOFF`, the line below which it will admit an inter-chain pair at all, so pale yellow marks where a residue's contacts stop being informative by ipSAE's own standard.

**Residues with no contacts are not painted.** Their value is undefined rather than zero, and colouring them would be showing a measurement that does not exist.

#### Views 3 and 5 answer different questions about the same residues

This is the reason both exist, and it is the distinction to hold on to:

- **View 3, ipSAE d0res** — *how confident is AlphaFold about the partner chain, seen from this residue?* Averaged over **every** partner residue with a sub-cutoff PAE, near or far.
- **View 5, pDockQ2 contact quality** — *how well placed are the contacts this residue actually makes?* Averaged over **only the partner residues it touches**.

A residue can be green in one and not the other. View 6 exists to find exactly those residues.

**What to look for.** Whether the greenest residues here are the same ones that were greenest in View 3.

**What a problem looks like.** Pale or orange residues in the middle of the contact patch: those are contacts AlphaFold does not place confidently. The reverse does not follow — a uniformly green view here does not guarantee a high pDockQ2, because the reported score is `mean interface pLDDT × mean contact ptm` pushed through a steep sigmoid (section 4.4), and a low-pLDDT interface holds it down however well the contacts are placed.

In [ ]:
# Same builder as View 3, same ramp, different array: pDockQ2's per-residue
# mean ptm instead of per-residue ipSAE_d0res.
render_view('pdockq2_value',
            lambda: ciu.build_interface_value_view(_source, chains, contacts,
                                                   _ptm_x, _ptm_y),
            lambda: ciu.value_ramp_legend('mean ptm at contacts'))

### View 6 — pDockQ2 contact quality against ipSAE d0res

**What is drawn.** Interface residues only, over a near-white cartoon, sorted into four categories by the two questions Views 3 and 5 ask separately. A residue with no contacts has no pDockQ2 value to compare against, so it is left unpainted and counted in the legend's last row.

**The two tests, and where their numbers come from.**

- **Confident about the partner chain:** `ipSAE_d0res ≥ 0.60`, from `THRESHOLDS['ipsae_d0res'].amber` — the same AFDB release edge View 4 uses.
- **Own contacts well placed:** mean contact `ptm ≥ 0.50`, which at the fixed `d0 = 10` is exactly **mean contact PAE better than 10 Å**. Not a new number either: 10 Å is ipSAE's own `PAE_CUTOFF`, so this test reads *"this residue's contacts beat the standard ipSAE uses to admit a pair at all"*.

**What the colours encode.**

| colour | category | meaning |
|---|---|---|
| bluish green `#009E73` | both agree, good | contacts well placed **and** confident about the partner chain |
| blue `#0072B2` | pDockQ2 only | own contacts well placed, but not confident about the partner chain as a whole |
| yellow `#F0E442` | ipSAE only | confident about the partner chain, but own contacts among the worse-placed |
| dark red `#A02020` | both agree, poor | neither test passes |
| — | not drawn | no contacts, so nothing to compare |

Green shares its colour with View 4's *confirmed contact*, deliberately: the two agreement views are meant to read as one family.

**Expect the yellow category to be empty, and read that as a result rather than a bug.** Across the four AFDB entries this notebook was developed against, no residue ever landed in "ipSAE only". Not one interface residue with `ipSAE_d0res ≥ 0.60` had a mean contact PAE worse than **2.7 Å**, against a line drawn at 10 Å; the lowest per-residue contact `ptm` seen anywhere across the four was **0.936**, against a cutoff of 0.50. That is worth stating plainly because the opposite is the natural assumption:

> **pDockQ2 is not stricter than ipSAE at the residue level.** Its reputation for strictness comes from the pooling — mean interface pLDDT multiplied by mean contact `ptm`, pushed through a steep sigmoid — and not from the quality of the contacts themselves.

So if your headline pDockQ2 is low while ipSAE is high, this view will usually show you that the contacts are fine and the arithmetic is happening elsewhere: in the pLDDT term, or in the sigmoid's steepness near its midpoint.

**Blue is the informative category.** Those residues touch the partner chain competently but are not confident about it overall — which the mechanism explains directly: `ipSAE_d0res` averages over the whole partner chain, so a residue whose few contacts are well placed can still be pulled below 0.60 by every other sub-cutoff pair in that row.

**The two chains can look completely different, and that is not a rendering fault.** Both signals are directional, and each chain is painted with its own direction's measurement. A large directional spread in the comparison at the end of Section 4 shows up here as one chain full of confirmed residues and the other with almost none.

In [ ]:
render_view('pdockq2_agreement',
            lambda: ciu.build_pdockq2_agreement_view(_source, chains, contacts,
                                                     _ptm_x, _ptm_y,
                                                     _ipsae_x, _ipsae_y),
            lambda: ciu.pdockq2_agreement_legend(contacts, _ptm_x, _ptm_y,
                                                 _ipsae_x, _ipsae_y,
                                                 short_x, short_y))

---
## Section 7 — Diagnostic Summary

**What we do here:** Bring together all five scores into a single interpretable diagnostic.

**Traffic-light thresholds** (based on AFDB filtering criteria and literature):
- **Green (high confidence):** ipSAE_d0res ≥ 0.6, ipTM ≥ 0.6, pDockQ ≥ 0.23, pDockQ2 ≥ 0.15, LIS ≥ 0.3
- **Amber (moderate):** half of the above thresholds
- **Red (low confidence):** below amber

These thresholds are starting heuristics; see the references for more detail.

In [ ]:
descriptions = {
    'ipsae_d0res': 'Primary AFDB classifier. Inter-chain PAE, d0 from valid-pair count.',
    'ipsae_d0chn': 'Most permissive variant. d0 from the full chain-pair length.',
    'ipsae_d0dom': 'Intermediate variant. d0 from domain-level residue count.',
    'iptm_d0chn':  'Global inter-chain confidence. All PAE cells used, no cutoff.',
    'pdockq':      'Structural plausibility. Contact count x interface pLDDT. No PAE.',
    'pdockq2':     'Contact PAE + interface pLDDT. Bridges pDockQ and ipSAE.',
    'lis':         'Density of inter-chain PAE < 12. Quantity of interaction.',
}
colour_hex = {'green': '#4CAF50', 'amber': '#FF9800', 'red': '#F44336'}

# Bands and colours come from `ciu.THRESHOLDS` via `ciu.traffic_light`, the one
# canonical table. ipSAE_d0res carries AlphaFold DB's four published band names;
# the other six use the three-colour scheme.
rows = ''
for _name, _value in scores.items():
    _colour, _band = ciu.traffic_light(_value, _name)
    rows += (
        f'<tr>'
        f'<td style="font-weight:bold;padding:6px 12px;">'
        f'{ciu.SCORE_DISPLAY_NAMES[_name]}</td>'
        f'<td style="padding:6px 12px;text-align:center;font-size:1.15em;font-weight:bold;">'
        f'{_value:.4f}</td>'
        f'<td style="padding:6px 12px;text-align:center;">'
        f'<span style="background:{colour_hex[_colour]};color:white;padding:3px 10px;'
        f'border-radius:12px;font-weight:bold;font-size:0.85em;">{_band}</span></td>'
        f'<td style="padding:6px 12px;font-size:0.88em;color:#555;">'
        f'{descriptions[_name]}</td>'
        f'</tr>'
    )

display(HTML(
    f'<h3>Confidence Score Summary — {ACCESSION_ID}</h3>'
    f'<p style="font-family:sans-serif;color:#555;margin:0 0 8px;">'
    f'{full_x}<br>{full_y}</p>'
    f'<table style="border-collapse:collapse;width:100%;font-family:sans-serif;">'
    f'<thead><tr style="background:#f5f5f5;">'
    f'<th style="padding:8px 12px;text-align:left;">Score</th>'
    f'<th style="padding:8px 12px;">Value</th>'
    f'<th style="padding:8px 12px;">Confidence</th>'
    f'<th style="padding:8px 12px;text-align:left;">What it measures</th>'
    f'</tr></thead><tbody>{rows}</tbody></table>'
))

In [ ]:
display(ciu.plot_score_agreement(scores))


In [ ]:
import textwrap

# The five independent values; the other two ipSAE variants are the same
# measurement under a more forgiving normalisation, not extra evidence.
score_names = list(ciu.AGREEMENT_SCORES)
lights = {name: ciu.traffic_light(scores[name], name) for name in score_names}

ipsae_val   = scores['ipsae_d0res']
pdockq_val  = scores['pdockq']
pdockq2_val = scores['pdockq2']
lis_val     = scores['lis']

tl_ipsae  = lights['ipsae_d0res'][0]
tl_pdockq = lights['pdockq'][0]
tl_lis    = lights['lis'][0]

n_green = sum(1 for name in score_names if lights[name][0] == 'green')
n_red   = sum(1 for name in score_names if lights[name][0] == 'red')

statements = []

if n_green >= 4:
    statements.append(
        'OVERALL: This complex has consistently HIGH confidence across all metrics. '
        'The interface is well-resolved (high pLDDT), structurally plausible (high pDockQ), '
        'and the PAE matrix shows strong inter-chain confidence.')
elif n_red >= 4:
    statements.append(
        'OVERALL: This complex has LOW confidence across most metrics. '
        'The predicted interaction may be unreliable. Treat structural conclusions with caution.')
else:
    statements.append(
        'OVERALL: Mixed confidence signals — scores disagree. See details below.')

# ipSAE high, pDockQ low
if tl_ipsae == 'green' and tl_pdockq in ('amber', 'red'):
    statements.append(
        'PAE vs STRUCTURE: AlphaFold is confident about relative chain positioning (PAE; '
        f'ipSAE={ipsae_val:.3f}), but there are few physical contacts at the interface '
        f'(pDockQ={pdockq_val:.3f}). This can occur when chains interact via a small, '
        'tight interface or when the predicted inter-chain distance is slightly too large '
        'for contacts to form under the 8 A cutoff.')

# pDockQ high, ipSAE low
if tl_pdockq == 'green' and tl_ipsae in ('amber', 'red'):
    statements.append(
        'STRUCTURE vs PAE: The interface has many contacts between well-resolved residues '
        f'(pDockQ={pdockq_val:.3f}), but AlphaFold PAE indicates uncertainty about the '
        f'relative chain arrangement (ipSAE={ipsae_val:.3f}). The local structure of each '
        'chain may be well-predicted even though the docking orientation is uncertain.')

# pDockQ2 vs pDockQ, each in units of its own canonical green threshold
pdockq_norm  = pdockq_val / ciu.THRESHOLDS['pdockq'].green
pdockq2_norm = pdockq2_val / ciu.THRESHOLDS['pdockq2'].green
if abs(pdockq_norm - pdockq2_norm) > 0.4:
    direction = 'pDockQ2 < pDockQ' if pdockq2_norm < pdockq_norm else 'pDockQ2 > pDockQ'
    statements.append(
        f'pDockQ vs pDockQ2 ({direction}): These two scores diverge, indicating that '
        'although physical contacts exist, the PAE confidence at those specific contact '
        'points is '
        + ('low' if pdockq2_norm < pdockq_norm else 'high') +
        '. pDockQ2 incorporates PAE at the interface and is the more informative of the two.')

# LIS high but ipSAE low
if tl_lis == 'green' and tl_ipsae in ('amber', 'red'):
    statements.append(
        f'LIS vs ipSAE: Many inter-chain PAE values are below {LIS_CUTOFF:.0f} A (LIS='
        f'{lis_val:.3f}), but when the cutoff is tightened to {PAE_CUTOFF:.0f} A and the '
        f'TM-score formula is applied (ipSAE={ipsae_val:.3f}), confidence drops. '
        'This suggests a broad but diffuse interaction rather than a tight, '
        'well-defined interface.')

print('=' * 65)
print('DIAGNOSTIC INTERPRETATION')
print('=' * 65)
for _statement in statements:
    print()
    print(textwrap.fill(_statement, width=63))
print()
print('─' * 65)
print('Score summary:')
for _name in score_names:
    print(f'  {ciu.SCORE_DISPLAY_NAMES[_name]:18s}: {scores[_name]:.4f}  '
          f'[{lights[_name][1]}]')
print('─' * 65)
print('Interface statistics:')
print(f'  Contact pairs     : {contacts.n_contact_pairs}')
print(f'  Interface res, {short_x}: {contacts.n_interface_residues_x}')
print(f'  Interface res, {short_y}: {contacts.n_interface_residues_y}')
print(f'  Mean pLDDT (if)   : {if_plddt.mean():.1f}')
print(f'  Low pLDDT (<70) if: {n_low_if} / {len(if_plddt)}')
print('=' * 65)
print()
print('References:')
print('  ipSAE : Dunbrack Lab (2025) biorxiv 2025.02.10.637595')
print('  pDockQ: Bryant et al. (2022) Nat Commun s41467-022-28865-w')
print('  pDockQ2: Zhu et al. (2023) Bioinformatics btad424')
print('  LIS   : Kim et al. (2024) biorxiv 2024.02.19.580970')
print('  AFDB bands / joint criterion: Han, Tsenkov, Venanzi et al. (2026) '
      'biorxiv 10.64898/2026.03.27.714458v2')
